# MAINTAIN AI V1.2 — Industrial Domain Bootstrap

V1.2 moves beyond C-MAPSS into equipment-relevant public telemetry. It uses one shared temporal encoder with category-conditioned input adapters and category heads. This notebook trains 24h/48h/7d future-risk labels where the public dataset supports timestamped events. Current automatic sources: MetroPT-3 compressor and CG-IPM-15 pump. Optional uploads can extend motor/conveyor categories later.

In [ ]:
!pip -q install pandas numpy scikit-learn torch pyarrow ucimlrepo

In [ ]:
import os, random, json, zipfile, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, torch
os.chdir('/content')
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))

## 1. Download compressor + pump data

MetroPT-3 is real multivariate compressor telemetry: 1,516,948 observations, 15 sensor features, sampled at 0.1 Hz, with documented failure intervals. The pump dataset is a six-month, 1-second synthetic industrial pump dataset with 26 sensor readings and failure codes.

In [ ]:
import requests, io
DATA=Path('/content/v1_2_data'); DATA.mkdir(exist_ok=True)

# MetroPT-3 UCI ZIP
metro_zip=DATA/'metropt3.zip'
if not metro_zip.exists():
    url='https://archive.ics.uci.edu/static/public/791/metropt3+dataset.zip'
    print('Downloading MetroPT-3...')
    urllib.request.urlretrieve(url,metro_zip)
with zipfile.ZipFile(metro_zip) as z: z.extractall(DATA/'metro')

# Pump source repository CSV
pump_csv=DATA/'MBG12_1sec_data.csv'
if not pump_csv.exists():
    url='https://raw.githubusercontent.com/rajesh-amrita/pump_maintenance/main/MBG12_1sec_data.csv'
    print('Downloading pump dataset...')
    urllib.request.urlretrieve(url,pump_csv)

print('Metro files:',[p.name for p in (DATA/'metro').rglob('*') if p.is_file()][:10])
print('Pump:',pump_csv, pump_csv.stat().st_size/1e6,'MB')

In [ ]:
# 2. Load and standardize compressor telemetry
metro_files=list((DATA/'metro').rglob('*.csv'))
if not metro_files: raise FileNotFoundError('MetroPT-3 CSV not found after extraction.')
metro_path=[p for p in metro_files if 'MetroPT3' in p.name or 'AirCompressor' in p.name]
metro_path=metro_path[0] if metro_path else metro_files[0]
metro=pd.read_csv(metro_path)
print('Metro shape:',metro.shape)
print('Metro columns:',metro.columns.tolist())

# UCI uses timestamp + 15 equipment signals.
metro['timestamp']=pd.to_datetime(metro['timestamp'])
metro=metro.sort_values('timestamp').reset_index(drop=True)

# Documented MetroPT-3 failure starts/ends (air-compressor failures).
failures=[
('2020-04-18 00:00:00','2020-04-18 23:59:00'),
('2020-05-29 23:30:00','2020-05-30 06:00:00'),
('2020-06-05 10:00:00','2020-06-07 14:30:00'),
('2020-07-15 14:30:00','2020-07-15 19:00:00')]
failure_starts=pd.to_datetime([x[0] for x in failures])

def future_risk(ts, hours):
    # 1 if a documented failure starts in the future horizon.
    delta=(failure_starts-ts).total_seconds()/3600.0
    return int(np.any((delta>=0)&(delta<=hours)))

for h in [24,48,168]: metro[f'risk_{h}h']=[future_risk(t,h) for t in metro['timestamp']]
metro['asset_id']='metropt3_compressor_01'; metro['category']='compressor'
print(metro[['timestamp','risk_24h','risk_48h','risk_168h']].head())

In [ ]:
# 3. Load and standardize pump telemetry
pump=pd.read_csv(pump_csv)
print('Pump shape:',pump.shape)
print('Pump columns:',pump.columns.tolist())

# Detect timestamp column.
time_candidates=[c for c in pump.columns if 'time' in c.lower() or 'date' in c.lower()]
if not time_candidates: raise ValueError('Pump dataset has no recognizable timestamp column.')
pump['timestamp']=pd.to_datetime(pump[time_candidates[0]],errors='coerce')
pump=pump.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)

fault_candidates=[c for c in pump.columns if 'failure' in c.lower() or 'fault' in c.lower()]
if not fault_candidates: raise ValueError('Pump dataset has no failure/fault column.')
fault_col=fault_candidates[0]
pump['fault_event']=pd.to_numeric(pump[fault_col],errors='coerce').fillna(0).gt(0).astype(int)
pump['asset_id']='cg_ipm15_pump_01'; pump['category']='pump'

# Build future event labels without using future values in the input window.
event_times=pump.loc[pump.fault_event==1,'timestamp'].drop_duplicates().sort_values().to_numpy(dtype='datetime64[ns]')
def pump_future_risk(ts,hours):
    t=np.datetime64(ts.to_datetime64())
    if len(event_times)==0: return 0
    idx=np.searchsorted(event_times,t,side='left')
    if idx>=len(event_times): return 0
    delta=(event_times[idx]-t)/np.timedelta64(1,'h')
    return int(0<=delta<=hours)
for h in [24,48,168]: pump[f'risk_{h}h']=[pump_future_risk(t,h) for t in pump['timestamp']]
print('Pump fault rows:',int(pump.fault_event.sum()))
print(pump[['timestamp','risk_24h','risk_48h','risk_168h']].head())

## 4. Common equipment representation

Each source is mapped into 12 common physical signal slots plus 12 presence indicators. This prevents the model from treating a compressor pressure channel and a pump vibration channel as the same physical measurement merely because they occupy the same tensor index. The category embedding supplies the equipment-domain context.

In [ ]:
COMMON=['temperature','vibration','current','pressure_1','pressure_2','flow','speed','load','power','torque','voltage','state']

def common_matrix(df,category):
    n=len(df); x=np.zeros((n,12),dtype=np.float32); present=np.zeros((n,12),dtype=np.float32)
    def put(slot,col):
        if col in df.columns:
            v=pd.to_numeric(df[col],errors='coerce').fillna(0).to_numpy(dtype=np.float32)
            x[:,slot]=v; present[:,slot]=1
    cols={str(c).lower():c for c in df.columns}
    if category=='compressor':
        put(0,cols.get('oil_temperature','__missing__')); put(2,cols.get('motor_current','__missing__'))
        put(3,cols.get('tp2','__missing__')); put(4,cols.get('tp3','__missing__')); put(5,cols.get('caudal_impulses','__missing__'))
        put(11,cols.get('comp','__missing__')); put(10,cols.get('motor_current','__missing__'))
    elif category=='pump':
        aliases={0:['stator_temperature','bearing_temperature','temperature'],1:['vibration_x','vibration_y','vibration_z','vibration'],2:['phase1_current','current','motor_current'],3:['inlet_pressure','pressure'],4:['outlet_pressure','pressure'],5:['flow_rate','flow'],6:['motor_speed','speed','rpm'],7:['load'],8:['active_power','power'],9:['torque'],10:['phase1_voltage','voltage'],11:['failure_code']}
        for slot,names in aliases.items():
            for name in names:
                if name in cols: put(slot,cols[name]); break
    return np.concatenate([x,present],axis=1)

metro_x=common_matrix(metro,'compressor')
pump_x=common_matrix(pump,'pump')
print('Compressor representation:',metro_x.shape)
print('Pump representation:',pump_x.shape)

In [ ]:
# 5. Downsample to 1-minute windows and build 24-step sequences
SEQ=24
def prepare(df,x,category):
    z=df.copy(); z['_row']=np.arange(len(z)); z=z.set_index('timestamp')
    numeric=pd.DataFrame(x,index=df['timestamp'])
    numeric=numeric.resample('1min').mean().interpolate(limit=3).ffill().bfill()
    labels=df.set_index('timestamp')[['risk_24h','risk_48h','risk_168h']].resample('1min').max().reindex(numeric.index).fillna(0)
    # Use one-minute samples; 24 steps = 24 minutes for these industrial bootstrap sources.
    values=numeric.to_numpy(dtype=np.float32)
    # normalize within source using past-independent global training statistics later; retain raw here.
    return numeric,labels

mvals,mlab=prepare(metro,metro_x,'compressor')
pvals,plab=prepare(pump,pump_x,'pump')
print('Minute rows:',len(mvals),len(pvals))

# Fit normalization using the earliest 70% of each source only.
norm_parts=[]
for v in [mvals,pvals]: norm_parts.append(v.iloc[:int(len(v)*0.7)].to_numpy())
fit=np.concatenate(norm_parts,axis=0)
mean=np.nanmean(fit,axis=0); std=np.nanstd(fit,axis=0); std=np.where(std<1e-8,1,std)

def sequences(values,labels,category_id,asset_prefix):
    a=((values.to_numpy(dtype=np.float32)-mean)/std)
    y=labels.to_numpy(dtype=np.float32)
    X=[]; Y=[]; ids=[]
    for i in range(SEQ-1,len(a)):
        X.append(a[i-SEQ+1:i+1]); Y.append(y[i]); ids.append(f'{asset_prefix}_{i}')
    return np.stack(X),np.stack(Y),ids

mx,my,mi=sequences(mvals,mlab,0,'compressor')
px,py,pi=sequences(pvals,plab,1,'pump')
X=np.concatenate([mx,px]); Y=np.concatenate([my,py]); C=np.concatenate([np.zeros(len(mx),dtype=np.int64),np.ones(len(px),dtype=np.int64)])
print('Combined:',X.shape,Y.shape,C.shape)
print('Positive rates:',Y.mean(axis=0))

In [ ]:
# 6. Asset/time-safe split: first 70% time for training, next 15% validation, final 15% test per category
def split_time(n):
    a=np.arange(n); return a[:int(.70*n)],a[int(.70*n):int(.85*n)],a[int(.85*n):]
mtr,mv,mt=split_time(len(mx)); ptr,pv,pt=split_time(len(px))
train_idx=np.concatenate([mtr, len(mx)+ptr]); val_idx=np.concatenate([mv,len(mx)+pv]); test_idx=np.concatenate([mt,len(mx)+pt])
Xt,Xv,Xte=X[train_idx],X[val_idx],X[test_idx]; Yt,Yv,Yte=Y[train_idx],Y[val_idx],Y[test_idx]; Ct,Cv,Cte=C[train_idx],C[val_idx],C[test_idx]
print('Train/val/test:',len(Xt),len(Xv),len(Xte))

In [ ]:
# 7. Shared category-aware temporal model
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SharedIndustrialModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.category_embedding=nn.Embedding(5,16)
        self.input_adapter=nn.ModuleList([nn.Sequential(nn.Linear(24,48),nn.GELU(),nn.Linear(48,64)) for _ in range(5)])
        self.conv=nn.Sequential(nn.Conv1d(64,96,5,padding=2),nn.BatchNorm1d(96),nn.GELU(),nn.Conv1d(96,128,5,padding=2),nn.BatchNorm1d(128),nn.GELU())
        self.gru=nn.GRU(128,128,2,batch_first=True,dropout=.1)
        self.fuse=nn.Sequential(nn.Linear(144,128),nn.GELU(),nn.Dropout(.1))
        self.risk=nn.Linear(128,3)
    def forward(self,x,c):
        adapted=torch.stack([self.input_adapter[int(ci)](x[i]) for i,ci in enumerate(c)],0)
        h=adapted.transpose(1,2); h=self.conv(h).transpose(1,2); h,_=self.gru(h); last=h[:,-1]
        z=self.fuse(torch.cat([last,self.category_embedding(c)],1)); return self.risk(z)

def loader(x,c,y,bs=256,shuffle=False): return DataLoader(TensorDataset(torch.tensor(x,dtype=torch.float32),torch.tensor(c),torch.tensor(y,dtype=torch.float32)),batch_size=bs,shuffle=shuffle)
tl=loader(Xt,Ct,Yt,256,True); vl=loader(Xv,Cv,Yv,512,False)
model=SharedIndustrialModel().to(device); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)
print('Parameters:',sum(p.numel() for p in model.parameters()))

In [ ]:
# 8. Train multi-horizon future-risk bootstrap
best=float('inf'); best_state=None; stale=0; patience=8; history=[]
for epoch in range(1,41):
    model.train(); total=n=0
    for bx,bc,by in tl:
        bx,bc,by=bx.to(device),bc.to(device),by.to(device); opt.zero_grad(set_to_none=True); logits=model(bx,bc)
        # BCE with logits for 24h/48h/7d; positive weighting is computed from training labels.
        loss=nn.functional.binary_cross_entropy_with_logits(logits,by)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); total+=loss.item()*len(bx); n+=len(bx)
    model.eval(); vt=vn=0
    with torch.no_grad():
        for bx,bc,by in vl:
            loss=nn.functional.binary_cross_entropy_with_logits(model(bx.to(device),bc.to(device)),by.to(device)); vt+=loss.item()*len(bx); vn+=len(bx)
    tr=total/n; va=vt/vn; history.append([epoch,tr,va]); print(f'Epoch {epoch:02d} | train={tr:.5f} | val={va:.5f}')
    if va<best-1e-4: best=va; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
    else: stale+=1
    if stale>=patience: print('Early stopping'); break
model.load_state_dict(best_state); model.eval()

In [ ]:
# 9. Evaluate and export V1.2 industrial bootstrap
from sklearn.metrics import roc_auc_score,average_precision_score
def evaluate(x,c,y):
    dl=loader(x,c,y,512,False); probs=[]; ys=[]
    with torch.no_grad():
        for bx,bc,by in dl: probs.append(torch.sigmoid(model(bx.to(device),bc.to(device))).cpu().numpy()); ys.append(by.numpy())
    p=np.concatenate(probs); yy=np.concatenate(ys); out={}
    for j,h in enumerate([24,48,168]):
        key=f'{h}h'; out[key]={}
        if len(np.unique(yy[:,j]))>1:
            out[key]['roc_auc']=float(roc_auc_score(yy[:,j],p[:,j])); out[key]['pr_auc']=float(average_precision_score(yy[:,j],p[:,j]))
        else: out[key]['roc_auc']=None; out[key]['pr_auc']=None
    return out
results={'model_version':'shared-industrial-v1.2','validation_loss':best,'test':evaluate(Xte,Cte,Yte),'normalization':'earliest-70%-per-source','sequence_steps':24,'step_minutes':1,'categories_present':['compressor','pump'],'risk_horizons':['24h','48h','7d']}
print(json.dumps(results,indent=2))
ART=Path('/content/maintain_ai_shared_industrial_v1_2'); ART.mkdir(exist_ok=True)
torch.save({'model_state_dict':model.state_dict(),'model_version':'shared-industrial-v1.2','input_features':24,'category_count':5,'categories':['induction_motor','pump','compressor','conveyor','other'],'risk_horizons':['24h','48h','7d'],'normalization_mean':mean.tolist(),'normalization_std':std.tolist(),'sequence_steps':24,'step_minutes':1},ART/'shared_industrial_v1_2.pt')
json.dump(results,open(ART/'evaluation.json','w'),indent=2)
json.dump({'common_features':COMMON,'sources':['MetroPT-3','CG-IPM-15 pump'],'risk_horizons':[24,48,168]},open(ART/'schema.json','w'),indent=2)

import shutil
bundle='/content/maintain_ai_shared_industrial_v1_2.zip'; shutil.make_archive('/content/maintain_ai_shared_industrial_v1_2','zip',ART)
print('Bundle:',bundle)

In [ ]:
from google.colab import files
files.download('/content/maintain_ai_shared_industrial_v1_2.zip')

## V1.2 scope

This is an equipment-domain bootstrap, not the final MAINTAIN AI production model. MetroPT-3 provides real compressor telemetry and documented failure intervals; the pump source provides labeled fault events. The shared encoder is category-aware, but motor/conveyor categories are not claimed as trained until their datasets are actually included. Production calibration still requires MAINTAIN AI telemetry and technician-confirmed outcomes.